# Individual Database Profiling Report: `TMP_REAN_DF2`

--- 
## 1. Introduction

This document serves as the standardized Data Profiling Report for the `TMP_REAN_DF2` database, a highly specialized component of the Teotihuacan Mapping Project's (TMP) legacy data archive. This analysis is situated within Phase 1 of the Digital TMP project, a foundational stage dedicated to the systematic, quantitative evaluation of legacy database architectures. The core purpose of this report is to provide a deep-dive analysis of the `TMP_REAN_DF2` schema by visualizing and interpreting a comprehensive suite of pre-computed metrics. By presenting this granular, empirical evidence in a structured and reproducible format, this report establishes a baseline understanding of this database's unique structural complexity, data quality, and analytical performance.

This report is a visualization and interpretation layer for a standardized set of metrics generated by an automated data profiling pipeline, as defined in the Phase 1 project plan. It does not represent a live analysis but rather a reproducible summary of the database's state, ensuring that the evaluation is consistent with the analyses of five other legacy and benchmark databases examined in this phase. The analysis will proceed systematically, beginning with a high-level schema overview and complexity assessment, followed by detailed table-level and column-level analyses, and concluding with an evaluation of performance on canonical analytical queries. The findings presented herein will serve as foundational evidence for the high-level comparative analysis and, ultimately, for the final Phase 1 White Paper, which will present a formal, evidence-based recommendation for a strategic architectural redesign of the unified TMP database in Phase 2.

--- 
## 2. Background

### 2.1. Context and Motivation

The primary goal of Phase 1 of the Digital TMP project is to conduct a systematic and quantitative evaluation of four legacy databases (`TMP_DF8`, `TMP_DF9`, `TMP_DF10`, `TMP_REAN_DF2`) and two modern benchmark databases. As outlined in the project's architectural and planning documents, this evaluation is a foundational step designed to generate the empirical evidence required to inform a strategic architectural redesign in Phase 2. The existing legacy databases, developed over several decades, exhibit a range of structural complexities and performance characteristics that must be rigorously measured and compared before a new, unified schema can be designed.

This report, focused specifically on `TMP_REAN_DF2`, is one of six standardized analyses that provide the granular evidence needed for this high-level comparison. By applying a consistent suite of profiling metrics to each database, the project can move beyond anecdotal or theoretical assessments of their respective strengths and weaknesses. This systematic, data-driven approach is critical for justifying the project's final architectural recommendations based on quantitative evidence rather than purely on theoretical principles. The findings from this individual analysis, when synthesized with those from its counterparts, will form the basis of a defensible, evidence-based strategy for building a performant, usable, and maintainable unified database for future Teotihuacan research.

### 2.2. Data: The `TMP_REAN_DF2` Database

The `TMP_REAN_DF2` database, commonly referred to as the REANs (Ceramic Reanalysis) database, is a distinct and critical component of the TMP digital legacy. Its creation, spanning from the early 1970s to the 1980s under the direction of George Cowgill and Evelyn Rattray, was motivated by the recognized analytical limitations of the original ceramic data in `DF8` and `DF9`. As documented in the *TMP DB Genealogy v2* and the *Technical Report on REANs Data and Methods*, the original analyses omitted a great deal of crucial information on vessel forms and decorative modes. The REANs project was initiated to capture this missing detail and apply a more refined chronological framework. The `TMP_REAN_DF2` version represents the culmination of decades of Curation, particularly the migration into an MS Access format by Ian Robertson under a 1999-2002 NSF grant.

The content of `TMP_REAN_DF2` is exclusively focused on providing a highly detailed, attribute-rich re-tabulation of the ceramic artifacts from the TMP surface collections. With over 240 ceramic-specific variables, it records granular counts for specific vessel forms (e.g., `ollapatl`), decorative techniques (`rtoincised`), and ware types (`lustrous`), far exceeding the simple phase totals that characterized its predecessors. This detail is invaluable for advanced ceramic analysis but introduces significant complexity. A crucial and defining feature of the REANs database is its **unit of analysis**: it is based on the original, unmerged field collection lots (~5,500), not the ~5,046 merged "sites" or "cases" used in `DF8` and `DF9`. This fundamental structural difference created a major, persistent challenge for data integration, with historical documents noting that approximately 300 "problematic collections" could not be reliably reconciled with `DF9`.

The database's structure mirrors that of the early `TMP_DF8`, employing a **vertical partitioning** model. The schema consists of **13 tables**, with a central `REAN_00` table acting as the hub and the remaining 12 tables (`REAN_01`, `REAN_02`, etc.) serving as thematic spokes linked by the common `ssn` key. This fragmented design, visualized in the project's ERD appendix, requires numerous joins to assemble a complete ceramic profile for a given collection.

The `TMP_REAN_DF2` database is also fraught with well-documented data quality issues. As detailed in the *Technical Report on Unfinished Core Database Work*, the decades-long, multi-analyst nature of the project led to inconsistencies from evolving classification criteria and analyst-specific practices (the "Pedro & Ceferino Subversion Factor"). The most significant problem was the undocumented removal of diagnostic artifacts to "specials" collections, leading to potential undercounts and major discrepancies between REANs and `DF9` totals. These historical complexities make the `TMP_REAN_DF2` an exceptionally powerful but challenging dataset to use responsibly.

### 2.3. Methods: Database Profiling Metrics

The analysis presented in this report is based on a standardized suite of pre-computed metrics generated by the `02_run_profiling_pipeline.py` script, as outlined in the Phase 1 project plan. This methodological approach ensures that the evaluation of `TMP_REAN_DF2` is both reproducible and directly comparable to the analyses of the other five databases under review. The metrics were calculated from a live PostgreSQL instance of the database and saved to disk as discrete JSON and CSV files. This notebook serves as the visualization and interpretation layer for this static, pre-computed data, rather than performing a live analysis. This ensures that the findings reported here are a stable, verifiable snapshot of the database's characteristics at the time the pipeline was executed.

The profiling pipeline gathers data across several distinct categories to provide a holistic assessment of the database. The report will present findings from each of these categories in sequence. **Schema-level metrics** provide a high-level overview, including aggregate object counts (e.g., table count) and total database size. **Table-level metrics** offer a more granular view of individual tables, assessing their size and row counts. **Column-level analysis** provides the deepest insights, examining both the structure (data types, nullability) and content (NULL value percentages, cardinality) of every column. Finally, this report presents custom **interoperability scores** designed to heuristically measure relational complexity, as well as **performance benchmarks** that measure query latency on a set of canonical analytical workloads. This standardized suite of metrics provides the quantitative foundation for the rigorous, evidence-based comparison across all Phase 1 databases.

### 2.4. Hypotheses

Based on the documented history and structure of the `TMP_REAN_DF2` database, this analysis is guided by a central hypothesis regarding its architectural efficiency. The database employs a vertically partitioned schema of 13 tables, a design similar to `TMP_DF8`, which requires joins across multiple tables to assemble a complete record. Given this structural fragmentation, and the join-intensive nature of the canonical queries defined in `canonical_queries_rean_df2.sql`, it is hypothesized that `TMP_REAN_DF2` will exhibit a significant performance penalty on analytical queries relative to a simple baseline scan. This expected latency will provide quantitative evidence that, despite its rich content, the database's legacy architecture is suboptimal for modern, high-performance analytical environments, reinforcing the need for a denormalized design in Phase 2.

---
## 3. Setup and Configuration

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import SVG, Markdown, display
from plotly.subplots import make_subplots

# --- CONFIGURATION ---------------------------------------------------
# SET THIS VARIABLE to the name of the database you want to analyze.
# e.g., 'TMP_DF8', 'TMP_DF9', 'tmp_benchmark_wide_numeric', etc.
DATABASE_NAME = "TMP_REAN_DF2"  # <--- CHANGE THIS
# ---------------------------------------------------------------------

# --- Path Definitions ---
# Use relative paths from the notebook's location in reports/individual_db_analysis/
METRICS_DIR = Path("../../outputs/metrics")
ERDS_DIR = Path("../../outputs/erds")

# --- Styling and Display Options ---
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)


def display_header(title):
    display(Markdown(f"### {title}"))


def load_metric_file(metric_name, file_type="csv"):
    """Helper function to safely load a metric file."""
    file_path = METRICS_DIR / f"{DATABASE_NAME}_{metric_name}.{file_type}"
    if not file_path.exists():
        print(f"⚠️ WARNING: Metric file not found: {file_path.name}")
        return None
    if file_type == "csv":
        return pd.read_csv(file_path)
    elif file_type == "json":
        with open(file_path, "r") as f:
            return json.load(f)


print(f"✅ Setup complete. Analyzing database: '{DATABASE_NAME}'")
print(f"Metrics Directory: {METRICS_DIR}")
print(f"ERD Directory: {ERDS_DIR}")

✅ Setup complete. Analyzing database: 'TMP_REAN_DF2'
Metrics Directory: ..\..\outputs\metrics
ERD Directory: ..\..\outputs\erds


---
## 4. Data Loading

In [ ]:
# Load all metric files into variables
basic_metrics = load_metric_file("basic_metrics", "json")
schema_counts = load_metric_file("schema_counts", "json")
interop_metrics = load_metric_file("interop_metrics", "json")

# Load table metrics and convert to DataFrame
table_metrics_data = load_metric_file("table_metrics", "json")
table_metrics_df = pd.DataFrame(table_metrics_data) if table_metrics_data else None

# Load column structure and convert to DataFrame
column_structure_data = load_metric_file("column_structure", "json")
column_structure_df = (
    pd.DataFrame(column_structure_data) if column_structure_data else None
)

# Load column profiles and convert to DataFrame
column_profiles_data = load_metric_file("column_profiles", "json")
column_profiles_df = (
    pd.DataFrame(column_profiles_data) if column_profiles_data else None
)

# Performance benchmarks remain as CSV
performance_df = load_metric_file("performance_benchmarks")

print("✅ Data loading complete.")

✅ Data loading complete.


---
## 5. High-Level Overview & Schema Visualization

### 5.1. Data: Database and Schema-Level Metrics

This section presents a high-level, aggregate analysis of the `TMP_REAN_DF2` database, providing a "30,000-foot view" of its overall size, composition, and structural complexity. The metrics presented below are schema-wide statistics, sourced from the `TMP_REAN_DF2_basic_metrics.json`, `TMP_REAN_DF2_schema_counts.json`, and `TMP_REAN_DF2_interop_metrics.json` files generated by the profiling pipeline. These summary statistics offer a quantitative starting point for assessing the database's architecture before proceeding to more granular table and column-level analyses.

The summary table will present key metrics that quantify the database's scale and relational complexity. `table_count` is a direct measure of structural fragmentation, representing the total number of user-defined tables within the schema. `database_size_mb` quantifies the total disk space consumed by the database. The remaining metrics are custom heuristic scores designed to measure relational complexity: the **Join Dependency Index (JDI)** measures the density of formal foreign key relationships; the **Logical Interoperability Factor (LIF)** assesses the potential for *ad hoc* joins based on column name and data type similarity; and the **Normalization Factor (NF)** provides a composite score to estimate the overall degree of schema normalization.

### 5.2. Theory & Methods: Schema Complexity and Interoperability Metrics

The schema-level metrics presented in this section are derived from two sources: direct queries against the PostgreSQL `information_schema` catalog and a suite of custom heuristic scores designed to quantify architectural complexity. The combination of these methods provides a multi-faceted, quantitative assessment of the database schema.

**Schema Object Counts:** Fundamental metrics such as `table_count`, `view_count`, and `function_count` are derived from straightforward `COUNT` queries on the relevant tables within the `information_schema`. For example, the table count is obtained by querying `information_schema.tables` where the `table_schema` matches the target schema of the analysis. These direct counts provide a baseline measure of the number of discrete objects that comprise the database structure.

**Interoperability Metrics:** To move beyond simple counts, three custom heuristic metrics were developed to provide a more nuanced assessment of relational complexity. These are defined as follows:
*   **Join Dependency Index (JDI):** The JDI is a measure of relational complexity based on the density of defined foreign key relationships. It is calculated as `JDI = foreign_key_count / max_possible_foreign_keys`, where `max_possible_foreign_keys` is derived from the number of tables (`n`) as `n * (n - 1) / 2`. A JDI score closer to 1.0 indicates a schema with a dense web of explicit relationships, suggesting higher normalization, while a score closer to 0 indicates a schema with few formal relationships, typical of denormalized or fragmented designs.
*   **Logical Interoperability Factor (LIF):** The LIF is a heuristic designed to estimate the potential for *ad hoc* joins where formal foreign keys may not exist. It operates on the assumption that columns with similar names and data types across different tables are likely to represent the same logical entity and are thus joinable. The metric is calculated by counting the number of distinct column name/data type pairs that appear in more than one table, providing a rough measure of logical cohesion.
*   **Normalization Factor (NF):** The NF is a composite score that combines the `table_count` and the `JDI` into a single, normalized value between 0 and 1. This score provides a holistic heuristic for the degree of schema normalization or fragmentation. Higher values suggest a more normalized schema (many tables with dense relationships), while lower values indicate a simpler, denormalized, or fragmented structure. The primary value of these heuristic scores lies not in their absolute values, but in their utility for *relative comparison* across the different database schemas analyzed in Phase 1.

In [ ]:
display_header(f"Key Metrics for: {DATABASE_NAME}")

summary_data = {}
if basic_metrics:
    summary_data.update(basic_metrics)
if schema_counts:
    summary_data.update(schema_counts)
if interop_metrics:
    summary_data.update(interop_metrics)
if table_metrics_df is not None:
    summary_data["total_estimated_rows"] = int(table_metrics_df["row_estimate"].sum())

if summary_data:
    summary_series = pd.Series(summary_data).rename("Value").to_frame()
    display(summary_series)
else:
    print("No summary metrics available.")

### Key Metrics for: TMP_REAN_DF2

### 5.3. Results: Key Metrics for `TMP_REAN_DF2`

The high-level metrics for `TMP_REAN_DF2` profile a moderately complex and fragmented database. The schema consists of **13 tables** and occupies a total of **14 MB** of disk space. Its Join Dependency Index (JDI) is **0.1538**, and its composite Normalization Factor (NF) is **0.1857**. The Logical Interoperability Factor (LIF) is **1**, indicating very few opportunities for ad-hoc joins based on shared column names, apart from the primary `ssn` key. The table count and NF score place its structural complexity as significantly lower than `TMP_DF8` (27 tables, NF 0.2139) and `TMP_DF9` (62 tables, NF 0.3503), but substantially more complex than the single-table benchmark databases.

### 5.4. Discussion: Initial Assessment of Schema Complexity

The high-level schema metrics confirm that `TMP_REAN_DF2` employs a vertically partitioned architecture, a design pattern consistent with its historical counterpart, `TMP_DF8`. The presence of 13 tables, each containing a thematic subset of the detailed ceramic reanalysis data, is direct evidence of this fragmentation. The moderately low Normalization Factor (NF) of 0.1857 accurately reflects a schema that is neither a simple flat file nor a deeply relational, normalized system. It occupies a middle ground of moderate structural complexity.

The JDI score of 0.1538, while higher than that of `TMP_DF8`, still indicates a relatively low density of formal foreign key relationships. This confirms that, like `DF8`, the primary relationships are the logical links between the peripheral data tables and the central `REAN_00` table via the `ssn` key, rather than a complex web of interdependencies between the data tables themselves. This initial assessment points to a schema that, by design, will require multi-table joins to assemble a comprehensive ceramic record for any given collection unit, a characteristic that is hypothesized to create performance overhead for analytical queries.

### 5.5. Data & Methods: Entity-Relationship Diagram (ERD)

An Entity-Relationship Diagram (ERD) is a visual representation of a database schema that illustrates the entities (tables), their attributes (columns), and the relationships between them. It serves as a critical tool for understanding the logical structure of a database, revealing its degree of normalization, the nature of its data dependencies, and its overall architectural complexity.

The ERD presented in this report was not manually drawn but was generated through an automated process to ensure it provides a completely accurate and objective reflection of the live `TMP_REAN_DF2` database schema. The diagram was created by the `03_generate_erds.py` script, which uses the `sqlalchemy-schemadisplay` library to programmatically inspect the live database's metadata. This process automatically discovers all tables, columns, and foreign key constraints and uses the `graphviz` software toolkit to render a graphical representation of these objects and their relationships. This automated methodology guarantees that the ERD is a direct, empirical visualization of the database's actual structure, free from any idealization or interpretation.

In [ ]:
display_header(f"Full ERD for: {DATABASE_NAME}")

try:
    # Find the most recent ERD file for the database
    erd_files = sorted(ERDS_DIR.glob(f"{DATABASE_NAME}_full_ERD_*.svg"), reverse=True)
    if erd_files:
        display(SVG(erd_files[0]))
    else:
        print(f"❌ ERROR: Full ERD SVG file not found for '{DATABASE_NAME}'.")
except Exception as e:
    print(f"An error occurred while displaying the ERD: {e}")

### Full ERD for: TMP_REAN_DF2

### 5.6. Results & Discussion: Visualizing Relational Complexity

The Entity-Relationship Diagram for `TMP_REAN_DF2` provides a clear and unambiguous visualization of its vertically partitioned architecture. The diagram displays a distinct "hub-and-spoke" pattern, with the `REAN_00` table serving as the central hub that contains the primary site identifiers. All other 12 data tables (`REAN_01` through `REAN_fc_adds`) are depicted as spokes, each connected directly to the `REAN_00` hub via a foreign key relationship on the `ssn` column. Crucially, there are no relationship lines connecting any of the peripheral data tables to one another.

This visual evidence powerfully corroborates the quantitative metrics. The `table_count` of 13 is immediately apparent. The "hub-and-spoke" layout perfectly explains the moderate JDI score; the schema has formal relationships, but they all follow a simple, repetitive pattern back to a single central table rather than forming a complex, interconnected web. The diagram thus serves as a definitive illustration of a schema that is partitioned, not deeply relational. It makes clear that any query requiring a comprehensive view of the ceramic data—for example, comparing phase totals from `REAN_01` with specific bowl types from `REAN_08`—will necessitate multiple, costly join operations, providing strong visual support for the hypothesis that this fragmentation will negatively impact analytical performance.

---
## 6. Table-Level Analysis

### 6.1. Data: Table-Level Metrics

This section transitions from the high-level schema overview to a more granular analysis of the individual tables within the `TMP_REAN_DF2` database. The data presented here are sourced from the `TMP_REAN_DF2_table_metrics.json` file, which contains a detailed statistical profile for each of the 13 tables. By examining these metrics, we can identify the largest and most data-rich tables, assess their general health, and understand how data is distributed across the vertically partitioned schema.

The upcoming summary tables and charts will present key metrics for each table. The `row_estimate` provides a statistical approximation of the number of rows, offering a quick measure of a table's scale. `column_count` indicates the width or number of attributes in each table. `total_size` reports the total disk space consumed by the table and all of its associated indexes, identifying which tables are the largest contributors to the database's storage footprint. Finally, `bloat_percent` is a critical database health indicator, quantifying the percentage of a table's file that consists of unused, reclaimable space.

### 6.2. Theory & Methods: Assessing Table Health and Size

The table-level metrics presented in this section are calculated using standard PostgreSQL functions and statistical queries that provide efficient and reliable information about table size and health. The methods used are designed to avoid performance-intensive operations while still yielding accurate assessments.

**Row Estimates:** The `row_estimate` for each table is not derived from an expensive `COUNT(*)` operation, which would require a full table scan. Instead, it is a highly efficient estimate sourced directly from the `pg_class.reltuples` column in PostgreSQL's internal statistics catalog. This value is updated by the `ANALYZE` command (and autovacuum daemon) and typically provides a very close approximation of the actual row count for static or infrequently updated tables, making it a standard and performant method for assessing table size.

**Table and Index Size:** The various size metrics (`table_size`, `index_size`, `total_size`) are calculated using built-in PostgreSQL functions such as `pg_relation_size()` and `pg_total_relation_size()`. These functions measure the actual disk space allocated to the table's data file (the "heap") and its associated indexes. The values are presented in a human-readable format (e.g., kB, MB) for easier interpretation.

**Table Bloat:** Table bloat refers to unused space that accumulates within a PostgreSQL table's data file due to the database's Multi-Version Concurrency Control (MVCC) implementation. When rows are updated (`UPDATE`) or deleted (`DELETE`), the old row versions are not immediately removed from the file; they are marked as "dead" and remain until a `VACUUM` process reclaims the space. `bloat_percent` is a key indicator of database health and is calculated using a standard community-provided SQL query that statistically estimates the amount of this dead space. High bloat percentages can negatively impact performance by increasing the number of disk pages that must be scanned to satisfy a query. It often suggests a need for database maintenance, such as running a `VACUUM FULL` operation or tuning autovacuum settings.

In [ ]:
display_header("Table Metrics Summary")

if table_metrics_df is not None and not table_metrics_df.empty:
    display(
        table_metrics_df.sort_values(
            by="row_estimate", ascending=False
        ).style.background_gradient(
            cmap="viridis", subset=["row_estimate", "bloat_percent"]
        )
    )
else:
    print("No table metrics data available.")

### Table Metrics Summary

### 6.3. Results: Table Metrics Summary for `TMP_REAN_DF2`

The table metrics for `TMP_REAN_DF2` reveal a remarkably consistent and uniform structure, reinforcing the vertical partitioning model. All 13 tables in the database share an identical row estimate of **5,055 rows**. This uniformity is a critical finding, confirming that each table represents a thematic slice of the same core set of 5,055 reanalyzed ceramic collections. The database is, in effect, a single wide logical entity that has been physically divided into 13 tables.

The tables are also very similar in size, reflecting a relatively even distribution of columns across the partitions. The `REAN_fc_adds` table, which contains metadata and flags about the reanalysis process, is the largest at **680 kB**. The main data tables, `REAN_01` through `REAN_08`, are all closely clustered in size, ranging from **528 kB** to **568 kB**. Bloat percentages are moderate and consistent, with most tables falling between **30% and 42%**. The smallest tables, `REAN_00` and `REAN_10`, show higher relative bloat percentages of 68.6% and 71.1% respectively, though the absolute amount of wasted space is comparable to the other tables.

In [ ]:
display_header("Largest Tables by Total Size and Bloat")

if table_metrics_df is not None and not table_metrics_df.empty:
    # Convert pretty size string to bytes for sorting
    def size_to_bytes(s):
        if not isinstance(s, str):
            return 0
        num, unit = s.split()
        num = float(num)
        if "KB" in unit:
            return num * 1024
        if "MB" in unit:
            return num * 1024**2
        if "GB" in unit:
            return num * 1024**3
        return num

    df_copy = table_metrics_df.copy()
    df_copy["total_bytes"] = df_copy["total_size"].apply(size_to_bytes)
    df_copy["bloat_bytes_val"] = df_copy["bloat_bytes"]

    top_10_size = df_copy.nlargest(10, "total_bytes")
    top_10_bloat = df_copy.nlargest(10, "bloat_bytes_val")

    # Display tables
    display(Markdown("**Top 10 Tables by Total Size**"))
    display(
        top_10_size[["table_name", "total_size", "row_estimate"]].reset_index(drop=True)
    )

    display(Markdown("**Top 10 Tables by Bloat Size**"))
    display(
        top_10_bloat[
            ["table_name", "bloat_size", "bloat_percent", "row_estimate"]
        ].reset_index(drop=True)
    )

    # Create subplots
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Top 10 Tables by Total Size", "Top 10 Tables by Bloat Size"),
    )

    fig.add_trace(
        go.Bar(
            y=top_10_size["table_name"],
            x=top_10_size["total_bytes"],
            orientation="h",
            name="Total Size",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(
            y=top_10_bloat["table_name"],
            x=top_10_bloat["bloat_bytes_val"],
            orientation="h",
            name="Bloat Size",
        ),
        row=1,
        col=2,
    )

    fig.update_layout(
        title_text=f"Table Size Analysis for {DATABASE_NAME}",
        height=500,
        showlegend=False,
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_xaxes(title_text="Size (Bytes)", row=1, col=1)
    fig.update_xaxes(title_text="Bloat (Bytes)", row=1, col=2)
    fig.show()
else:
    print("No table metrics data available for plotting.")

### Largest Tables by Total Size and Bloat

**Top 10 Tables by Total Size**

**Top 10 Tables by Bloat Size**

### 6.4. Results: Largest Tables by Size and Bloat

The analysis of the largest tables by total size and by bloat confirms the schema's homogeneity. The `REAN_fc_adds` and `REAN_01` tables are the largest, occupying 680 kB and 568 kB respectively. The remaining data tables are all very similar in scale.

A consistent pattern is observed in table bloat, where every single table has an estimated bloat size of **0.15 MB**. This remarkable uniformity in absolute bloat suggests that the database was likely created through a single, systematic bulk-loading process, and any subsequent data modifications or deletions were applied evenly across the entire dataset. The variation in `bloat_percent` is therefore simply an artifact of the different total sizes of the tables; the tables with fewer columns (and thus smaller total sizes) show a higher relative bloat percentage for the same absolute amount of wasted space.

### 6.5. Discussion: Identifying Key Tables and Health Concerns

The table-level analysis provides definitive, quantitative evidence of the `TMP_REAN_DF2` database's strict vertical partitioning architecture. The consistent row estimate of 5,055 across every single table is the most significant finding, proving that these tables are not independent entities but are fragments of a single, coherent dataset. Each table holds a different set of attributes for the same 5,055 ceramic reanalysis cases. This partitioned structure has direct and significant implications for query performance, as any comprehensive analysis will require joining many of these 5,055-row tables.

From a health perspective, the moderate and highly uniform bloat levels are not a major concern. The total database size is small (14 MB), meaning the performance impact of this bloat is negligible. The pattern of uniform bloat is more interesting as a historical artifact, strongly suggesting a systematic, one-time data loading and modification event. The primary takeaway from this table-level analysis remains the stark confirmation of the highly partitioned structure, which is the central architectural feature influencing the database's analytical usability.

---
## 7. Column-Level Analysis

### 7.1. Data Type Frequencies

#### 7.1.1. Data & Methods

This analysis examines the fundamental composition of the `TMP_REAN_DF2` schema by profiling the data types used across all of its columns. The data for this section is sourced from the `TMP_REAN_DF2_column_structure.json` file, which contains metadata for every column in the database, including its assigned PostgreSQL data type. The method involves aggregating this data to count the frequency of each distinct data type. The resulting distribution provides critical insight into the database's design philosophy, particularly regarding how different kinds of information (e.g., categorical, numeric, textual) are physically stored. This choice has direct implications for storage efficiency, data integrity, and analytical usability.

In [ ]:
display_header("Data Type Distribution")

if column_structure_df is not None:
    type_counts = column_structure_df["data_type"].value_counts().reset_index()
    type_counts.columns = ["data_type", "count"]

    # Calculate percentages
    type_counts["percentage"] = (
        type_counts["count"] / type_counts["count"].sum() * 100
    ).round(2)

    # Display comprehensive table
    display(Markdown("**Complete Data Type Distribution**"))
    display(type_counts.style.format({"percentage": "{:.2f}%"}))

    # Display summary statistics
    display(Markdown("**Data Type Summary**"))
    summary_stats = pd.DataFrame(
        {
            "Total Columns": [type_counts["count"].sum()],
            "Unique Data Types": [len(type_counts)],
            "Most Common Type": [
                f"{type_counts.iloc[0]['data_type']} ({type_counts.iloc[0]['count']} columns)"
            ],
            "Least Common Type": [
                f"{type_counts.iloc[-1]['data_type']} ({type_counts.iloc[-1]['count']} columns)"
            ],
        }
    )
    display(summary_stats)

    fig = px.bar(
        type_counts,
        x="data_type",
        y="count",
        title=f"Column Data Type Frequencies in {DATABASE_NAME}",
        labels={"count": "Number of Columns", "data_type": "Data Type"},
    )
    fig.show()
else:
    print("No column structure data available.")

### Data Type Distribution

**Complete Data Type Distribution**

**Data Type Summary**

,Total Columns,Unique Data Types,Most Common Type,Least Common Type
0,241,3,smallint (221 columns),boolean (9 columns)


#### 7.1.2. Results & Discussion

The analysis of column data types reveals a schema built almost exclusively on a single numeric type. Of the 241 columns in the `TMP_REAN_DF2` database, an overwhelming **221 columns (91.7%)** are defined as `smallint`. The remaining columns are a small number of `boolean` flags (9 columns, 3.7%) and `text` identifiers (11 columns, 4.6%).

This extreme dominance of the `smallint` data type is a direct consequence of the database's primary function: to store counts of ceramic artifacts. Nearly every column in the `REAN_01` through `REAN_10` tables represents a raw count of a specific ceramic type, form, or decorative mode. The `text` columns are used exclusively for identifiers (`site`, `unit`, etc.) and comments, while the `boolean` fields in the `REAN_fc_adds` table are flags tracking the status of the reanalysis process itself. Unlike its predecessors (`DF8`, `DF9`), `TMP_REAN_DF2` does not appear to use integer codes to represent categorical data; rather, it uses the integers to store true quantitative counts. This makes the database's content much more transparent, but the schema's "column-based artifact design"—where each of the 221 `smallint` columns represents a distinct artifact category—remains a significant structural feature that impacts usability.

### 7.2. Data Completeness: NULL Value Analysis

#### 7.2.1. Data & Methods

This analysis assesses data completeness and quality by measuring the prevalence of `NULL` values across every column in the `TMP_REAN_DF2` database. Sourced from the `TMP_REAN_DF2_column_profiles.json` file, the core metric is `null_percent`, which calculates the percentage of rows containing a `NULL` for each column. In standard SQL, `NULL` is the correct and conventional representation for missing or unknown data. A high percentage of `NULL` values in a column can indicate issues with the original data collection process or subsequent data entry, potentially impacting the reliability of any analysis that relies on that column. This analysis is therefore a critical measure of overall data quality and integrity.

In [ ]:
display_header("Top 20 Columns by Percentage of NULL Values")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Ensure we only show columns with NULLs
    null_df = column_profiles_df[column_profiles_df["null_percent"] > 0].copy()

    if not null_df.empty:
        # Create a full column identifier for clarity
        null_df["full_column_name"] = (
            null_df["tablename"].astype(str) + "." + null_df["column_name"].astype(str)
        )

        top_20_nulls = null_df.nlargest(20, "null_percent")

        # Display table
        display(Markdown("**Top 20 Columns with Highest NULL Percentages**"))
        table_display = top_20_nulls[
            [
                "full_column_name",
                "null_percent",
                "null_count_estimate",
                "row_count_exact",
            ]
        ].copy()
        table_display.columns = ["Column", "NULL %", "NULL Count", "Total Rows"]
        display(table_display.reset_index(drop=True))

        # Display summary statistics
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [len(null_df)],
                "Columns with 100% NULLs": [
                    len(null_df[null_df["null_percent"] == 100])
                ],
                "Average NULL %": [f"{null_df['null_percent'].mean():.2f}%"],
                "Median NULL %": [f"{null_df['null_percent'].median():.2f}%"],
            }
        )
        display(null_summary)

        fig = px.bar(
            top_20_nulls,
            y="full_column_name",
            x="null_percent",
            orientation="h",
            title=f"Top 20 Columns by NULL Percentage in {DATABASE_NAME}",
            labels={
                "null_percent": "Percentage of Rows that are NULL (%)",
                "full_column_name": "Column",
            },
        )
        fig.update_layout(height=600)
        fig.update_yaxes(autorange="reversed")
        fig.show()
    else:
        print("✅ Excellent! No columns with NULL values were found.")

        # Still show summary even when no NULLs
        display(Markdown("**NULL Value Summary**"))
        null_summary = pd.DataFrame(
            {
                "Total Columns Analyzed": [len(column_profiles_df)],
                "Columns with NULLs": [0],
                "Data Completeness": ["100% - Perfect!"],
            }
        )
        display(null_summary)
else:
    print("No column profile data available.")

### Top 20 Columns by Percentage of NULL Values

**Top 20 Columns with Highest NULL Percentages**

**NULL Value Summary**

,Total Columns Analyzed,Columns with NULLs,Columns with 100% NULLs,Average NULL %,Median NULL %
0,241,225,1,14.18%,7.02%


#### 7.2.2. Results & Discussion

The NULL value analysis of `TMP_REAN_DF2` reveals a high degree of data completeness, but also highlights areas of significant sparsity. The vast majority of the 241 columns contain some `NULL` values, with a median NULL percentage of **7.02%** across the dataset. This indicates that for a typical ceramic category, sherds of that type were not found in roughly 7% of the collection units.

The columns with the highest NULL percentages are highly informative. The `rean_month` column is 100% NULL, indicating this field was never populated. Several columns in the `REAN_aux_obs` table, which contains specialized observations, show over 95% NULLs (e.g., `to_total`, `xol_tto`, `tlajinga`), confirming that these were rare attributes only recorded for a small subset of collections. Similarly, the `comment_admin` and `comment_fc_adds` fields are almost entirely NULL, which is expected for optional text fields. This pattern of high NULL rates in specific, specialized columns is not necessarily a data quality issue; rather, it accurately reflects the rarity of certain ceramic types and observations in the archaeological record. It confirms that the database uses standard `NULL`s to correctly represent the absence of data, a significant improvement over the sentinel value practice of `DF8` and `DF9`.

### 7.3. Data Complexity: Cardinality Analysis

#### 7.3.1. Data & Methods

This section analyzes the complexity of the data within each column by measuring its **cardinality**. The data is sourced from the `TMP_REAN_DF2_column_profiles.json` file. Cardinality is defined as the number of unique or distinct values present in a column. This metric is a fundamental characteristic of a dataset and is critical for understanding the nature of each attribute.

The analysis of cardinality helps to distinguish between different types of columns. Columns with very high cardinality are typically identifiers or primary keys. Columns with very low cardinality (e.g., 2 to 10 distinct values) usually represent categorical variables or flags. By examining the distribution of cardinalities, we can gain insight into the database's structure, identify potential join keys, and understand the complexity of the data within its vertically partitioned schema.

In [ ]:
display_header("Column Cardinality Distribution")

if column_profiles_df is not None and not column_profiles_df.empty:
    # Create a full column identifier
    df = column_profiles_df.copy()
    df["full_column_name"] = df["tablename"] + "." + df["column_name"]

    # Display tables of highest and lowest cardinality columns
    display(Markdown("**Columns with Highest Cardinality (Most Unique)**"))
    display(
        df.nlargest(10, "distinct_values_estimate")[
            ["full_column_name", "distinct_values_estimate"]
        ]
    )

    display(Markdown("**Columns with Lowest Cardinality (Least Unique)**"))
    display(
        df[df["distinct_values_estimate"] > 1].nsmallest(
            10, "distinct_values_estimate"
        )[["full_column_name", "distinct_values_estimate"]]
    )

    # Create a histogram of cardinalities to see the distribution
    fig = px.histogram(
        df,
        x="distinct_values_estimate",
        log_y=True,
        title=f"Distribution of Column Cardinalities in {DATABASE_NAME}",
        labels={"distinct_values_estimate": "Number of Distinct Values (Cardinality)"},
    )
    fig.show()
else:
    print("No column profile data available.")

### Column Cardinality Distribution

**Columns with Highest Cardinality (Most Unique)**

**Columns with Lowest Cardinality (Least Unique)**

#### 7.3.2. Results & Discussion

The cardinality analysis of `TMP_REAN_DF2` confirms that the database is primarily composed of identifier columns and low-to-moderate complexity count data. As expected, the `ssn` columns in every table have the highest possible cardinality, with an estimated -1 distinct values, confirming their function as primary or foreign keys. The administrative text identifiers (`site`, `unit`, `subsite`) also show high cardinality with over 100 unique values each, reflecting the diversity of site names.

The most significant finding is the distribution of cardinalities for the 221 ceramic count columns. The vast majority of these columns have low to moderate cardinality. For example, `REAN_01.xolatot` has 253 unique count values, while many less common types like `REAN_06.polblackmete` or `REAN_09.tlalocvessel` have only 2-4 unique values (likely 0, 1, 2, etc.). The histogram of cardinalities shows that the overwhelming majority of columns have fewer than 50 distinct values. This distribution is a direct reflection of archaeological count data, where many artifact types are rare, and the number of items found per collection is typically low. It also reinforces the critique of the "column-based artifact design," as the database structure dedicates an entire column to a category that may only have a handful of non-zero entries across the entire dataset. This structural choice, while detailed, is inefficient from a storage and query perspective, providing further evidence for the need to transform this wide, sparse data into a more efficient format in Phase 2.

---
## 8. Performance Benchmark Analysis

### 8.1. Data, Theory & Methods

This section evaluates the analytical performance of the `TMP_REAN_DF2` database by measuring query execution latency on a set of standardized, canonical queries. The data for this analysis is sourced from the `TMP_REAN_DF2_performance_benchmarks.csv` file, which logs the results of these benchmark tests. The methodology is designed to provide a fair, reproducible, and representative test of the schema's efficiency under different analytical workloads.

The methodology involves executing a predefined set of three canonical queries against the database and measuring the time taken for each to complete, reported in milliseconds. These queries are not generic; they are hand-crafted and stored in the `phases/01_LegacyDB/sql/canonical_queries/canonical_queries_rean_df2.sql` file to specifically test the architectural characteristics of the `TMP_REAN_DF2` schema. The three queries represent distinct analytical workloads:
1.  **Baseline Scan:** A simple `COUNT(*)` on the main administrative table (`REAN_00`) to establish a baseline for raw I/O performance.
2.  **Multi-Table Join:** A query that joins the administrative table (`REAN_00`) with the main totals table (`REAN_01`) to simulate a typical analytical task requiring data from the vertically partitioned schema.
3.  **Complex Filtering:** A query that joins two tables and applies filtering conditions on attributes from both tables, representing a more complex but common analytical scenario.

By comparing the latency of the join-intensive queries to the baseline, we can quantitatively measure the performance impact of `TMP_REAN_DF2`'s fragmented design, directly testing the central hypothesis of this report.

In [ ]:
display_header("Canonical Query Performance Results")

if performance_df is not None and not performance_df.empty:
    display(performance_df[["query_name", "latency_ms", "status"]])

    # Plot the results for successful queries
    success_df = performance_df[performance_df["status"] == "Success"]
    if not success_df.empty:
        fig = px.bar(
            success_df,
            x="query_name",
            y="latency_ms",
            title=f"Query Latency for {DATABASE_NAME}",
            labels={"latency_ms": "Latency (ms)", "query_name": "Canonical Query"},
        )
        fig.show()
else:
    print("No performance benchmark data available.")

### Canonical Query Performance Results

### 8.2. Results: Query Performance for `TMP_REAN_DF2`

The performance benchmark results for `TMP_REAN_DF2` provide clear, quantitative data on the impact of its schema design on analytical query latency. The three canonical queries executed successfully, with the following recorded latencies:

*   **Baseline Performance - Query 1.1:** 0.91 ms
*   **Join Performance - Query 2.1:** 10.29 ms
*   **Complex Filtering - Query 3.1:** 2.14 ms

The baseline query, a simple count on a single table, executed almost instantaneously at just under 1 ms. The join performance query, which required joining two of the core 5,055-row tables (`REAN_00` and `REAN_01`), took **10.29 ms**. This represents a substantial performance degradation of **11.3 times** compared to the baseline scan. The complex filtering query, which also required a two-table join, was significantly faster but still took **2.14 ms**, representing a slowdown of **2.3 times** relative to the baseline. The most significant finding is the extreme latency penalty incurred by the simple two-table join, which directly measures the overhead imposed by the schema's fragmentation.

### 8.3. Discussion: Impact of Schema on Analytical Performance

The performance benchmark results strongly support the hypothesis that the vertically partitioned schema of `TMP_REAN_DF2` imposes a significant performance penalty on analytical queries. The fact that a simple two-table join was over **11 times slower** than the baseline scan provides direct, quantitative evidence of the overhead introduced by this fragmented design. This performance degradation is an inherent consequence of the architectural choice to store related attributes in separate tables. To fulfill the query, the database engine must perform costly join operations, linking data blocks from two distinct tables, which is far more computationally intensive than a simple scan of a single table.

While the absolute query times are low due to the small total size of the database, the *relative* performance degradation is the key finding. It empirically validates the argument that while partitioning may have been a logical organizational strategy in a legacy, file-based system, it creates an inherent performance bottleneck in a modern relational query environment. This finding provides further justification for the strategic goal of Phase 2: to create a denormalized, unified schema that minimizes the need for such joins, thereby ensuring an efficient and performant research environment.

---
## 9. Final Report and Recommendations

### 9.1. Summary of Results

This analysis provides a comprehensive, multi-faceted profile of the `TMP_REAN_DF2` legacy database, yielding a set of integrated findings that clearly characterize its architecture, data quality, and performance.

*   **Structural Complexity:**
    The `TMP_REAN_DF2` database is defined by a **vertically partitioned** architecture, fragmenting its 241 ceramic-specific variables across **13 tables**. The schema is visualized in the ERD as a distinct "hub-and-spoke" model, with 12 data tables linking to a central `REAN_00` table. This moderate complexity is quantified by a Normalization Factor (NF) of **0.1857**. The database is almost exclusively composed of `smallint` data types (**91.7% of all columns**), reflecting its primary function of storing raw ceramic counts. This design, while rich in detail, forces analysts to join numerous tables to assemble a complete ceramic profile.

*   **Data Quality & Health Concerns:**
    `TMP_REAN_DF2` demonstrates a significant improvement over `DF8` and `DF9` in its handling of missing data, using standard SQL `NULL`s instead of sentinel values. The analysis found that many specialized columns have high NULL percentages (some >95%), which accurately reflects the rarity of certain artifact types rather than a data quality issue. The database's unit of analysis is the original, unmerged field collection, a critical structural feature that, as documented in historical reports, creates significant challenges for integration with the merged-site structure of `DF9`. Table bloat is moderate and uniform across the schema, suggesting a systematic data loading history and posing no significant performance risk for this small (14 MB) database.

*   **Performance Profile:**
    The performance benchmarks quantitatively confirmed the negative impact of the database's partitioned structure on analytical queries. A standard two-table join query was **11.3 times slower** than a simple baseline scan of a single table (**10.29 ms** vs. **0.91 ms**). This dramatic performance degradation provides direct, empirical evidence that the schema's fragmentation creates a significant and measurable performance overhead for common analytical tasks.

### 9.2. Discussion

The most striking feature of the `TMP_REAN_DF2` database is the sheer **granularity and richness of its ceramic data**, a direct result of the multi-decade reanalysis project. With over 200 variables dedicated to specific ceramic forms, wares, and decorative modes, it represents a far more detailed dataset than the simple phase totals available in its predecessors. This detailed content is its greatest strength.

However, this rich content is contained within an **inefficient legacy architecture**. The database's primary characteristic is its vertical partitioning, a structural choice inherited from `TMP_DF8`. As with `DF8`, this results in a schema that is fragmented without being truly normalized in a modern sense. The analysis has shown this structure imposes a severe performance penalty on join-intensive queries. Furthermore, the database is saddled with significant historical baggage, including known data quality issues from inconsistent analytical criteria over time, and the major structural challenge of reconciling its collection-based records with the site-based records of `DF9`. It is a powerful but flawed dataset whose full potential is locked behind an inefficient and complex structure.

### 9.3. Conclusions & Implications for Phase 2 Redesign

Based on the comprehensive analysis of its structure, data quality, and performance, this report concludes that the `TMP_REAN_DF2` database, while containing invaluable and highly detailed ceramic data, is architecturally unsuitable for a modern analytical environment. Its partitioned design creates critical performance bottlenecks, and its historical inconsistencies require careful remediation.

*   **Based on this analysis, what are the key strengths and weaknesses of this database's design?**
    *   **Strengths:**
        *   **Rich, Granular Data:** The database contains an unparalleled level of detail on ceramic attributes, far exceeding any other TMP dataset. This is its single most important asset.
        *   **Standard NULL Usage:** Its use of standard SQL `NULL`s for missing data is a significant improvement in data integrity over `DF8` and `DF9`.
    *   **Weaknesses:**
        *   **Poor Join Performance:** The partitioned schema is computationally inefficient, leading to a **>11x slowdown** on a simple two-table join.
        *   **Structural Incompatibility:** Its collection-based unit of analysis is fundamentally incompatible with the site-based structure of `DF9`, posing a major integration challenge.
        *   **Known Data Quality Issues:** As documented extensively in project history, the data suffers from inconsistencies due to the long, multi-analyst reanalysis process and the problem of "specials" being removed from collections.

*   **What specific aspects of this schema should be preserved, changed, or discarded in the final unified database?**
    *   **Discard:**
        *   **Vertical Partitioning:** The core architectural strategy of splitting the ceramic data across 12 separate tables must be discarded.
        *   **The "Column-per-Artifact" Design:** The practice of dedicating a separate column for each of the 221 ceramic categories is a 1NF violation and must be abandoned.
    *   **Change:**
        *   **Structure:** The data should be transformed from its current "wide" and partitioned format into a "long" or tidy format. A new table structure with columns like `ssn`, `ceramic_type`, and `count` would be far more efficient and align with modern data analysis best practices.
        *   **Data Reconciliation:** The known discrepancies between REANs and `DF9` counts (due to "specials" and evolving criteria) must be investigated and, where possible, reconciled during the ETL process for the final unified database. The ~300 problematic collections require special attention.
    *   **Preserve:**
        *   **The Core Data Content:** The granular ceramic counts are the entire reason for this database's existence. This detailed information is a critical asset that must be preserved in its entirety and serve as the foundation for the ceramic component of the final unified database.
        *   **The `ssn` Foreign Key:** The `ssn` identifier, which links the ceramic data to a specific field collection, must be preserved as the essential key for joining this data to the main site attribute table in the final unified design.